# Dependencies + Imports

In [1]:
## dependencies: python 3.9.5; pip 21.1.1; gudhi 3.4.1
import gudhi
from IPython.display import Image
# add in the classes file
import sys 
sys.path.append('../Code/')
import classes as cl
# This reloads external modules every time a cell is run
# so we don't have to reboot kernel every time we modify classes.py
%load_ext autoreload
%autoreload 2
import csv #for loading function


# Definitions

In [2]:
def order_simplices(st):
    # defined before Classy Classes
    # takes a simplex tree (gudhi) as input 
    # assigns an integer to each simplex starting from the 0-simplices to the simplices of the highest dimension
    simplices = [s for (s, eps) in st.get_filtration()]
    ordering = []
    N = 0
    for k in range(st.dimension()+1):
        for s in simplices:
            if len(s) == k + 1:
                st.assign_filtration(s, float(N))
                ordering.append(s)
                N += 1
    ## ordered list of simplices from 0 to number of simplices: 
    simplices = [s for (s, eps) in st.get_filtration()]
    return simplices

def critical_values(F, dgvf): 
    ##input F a filtration, dgvf a discrete gradient vector field 
    ## Returns critical values in R^2 
    C = []
    for simplex in range(len(dgvf['C'])): 
        s = dgvf['C'][simplex]
#       this appends the simplex itself, but we want that simplex's coords in F!
        idx = simplices.index(s)
        C.append(bifiltration[idx])
#       The issue: we don't actually have a pairing between coordinates 
#       and simplices defined!! So there's nowhere to pull this data from. 
#       index = F.index(dgvf['C'][simplex])
#       print(index)
    #print(C)
    return (C)

## closing critical values under least upper bound 
def closure_lub(C): 
    C_bar = []
    for i in range(len(C)):
        c = C[i]
        C_bar.append(c)
        # the error is because we are trying to look through actual simplices
        # instead of their coords in the bifiltration.
        for j in range(i, len(C)):
            d = C[j]
            lub = (max(c[0],d[0]), max(c[1],d[1]))
            C_bar.append(lub)
    
    return list(set(C_bar))

# Initialize and import

In [3]:
# initialize simplex tree from Gudhi library
stt = gudhi.SimplexTree()
simplices_list = []

# importer function to run over multipersistence (.mprs) data file
data = []
# error checking
try: fd = open('../data/sample.mprs', 'r')
except OSError:
    print("Could not open/read file")
    sys.exit()
with fd:
    reader = csv.reader(fd, delimiter=':')
    num_entries = 0
    for row in reader:
        if not row:
            print("error! row not found!")
        num_entries = num_entries + 1
        data.append(list(row))
        datalist = []
    i = 0
    for x in data:
        tempfloats = [float(item) for item in data[i][1].split()]
        tempint = [int(item) for item in data[i][0].split()]
        templist = [tempint, tempfloats]
        datalist.append(templist)
        temp_simplex = cl.Simplex()
        temp_simplex.boundary = tempint
        temp_simplex.entrancevals = tempfloats
        temp_simplex.init_index = i
        stt.insert(temp_simplex.boundary) 
        simplices_list.append(temp_simplex)
        i += 1 
        
stt.get_filtration()
ordered_simplices = order_simplices(stt)

    
input_complex = cl.Complex(simplices_list, ordered_simplices, stt)

################################################## input file
#Discrete gradient vector field in a dictionary
# A_i paired with B_i
## TODO: write the gradient vector field with the indices of the simplices instead of the simplices themselves? 
V = {}
A = [[1],[3,1],[6,1],[2],[4],[6,4],[4,7],[5],[6],[7]]
B = [[0,1],[0,1,3],[1,3,6],[1,2],[2,4],[1,4,6],[4,6,7],[5,3],[6,5],[7,6]]
C = [[0],[3,0],[4,1],[3],[3,6]]
V['A'] = A
V['B'] = B
V['C'] = C
################################################
for a in A:
    for s in input_complex.simplices:
        if a == s.boundary:
            s.in_dgvf = 'A'
for b in B:
    for s in input_complex.simplices:
        if b == s.boundary:
            s.in_dgvf = 'B'
for c in C:
    for s in input_complex.simplices:
        if c == s.boundary:
            s.in_dgvf = 'C'
            s.criticality = True
            
input_complex.print_all_simplices()
for s in input_complex.simplices:
    print(s.entrancevals)

boundary: [0] critical? True in dgvf C
boundary: [1] critical? False in dgvf A
boundary: [2] critical? False in dgvf A
boundary: [3] critical? True in dgvf C
boundary: [4] critical? False in dgvf A
boundary: [5] critical? False in dgvf A
boundary: [6] critical? False in dgvf A
boundary: [7] critical? False in dgvf A
boundary: [0, 1] critical? False in dgvf B
boundary: [1, 2] critical? False in dgvf B
boundary: [2, 4] critical? False in dgvf B
boundary: [4, 7] critical? False in dgvf A
boundary: [7, 6] critical? False in dgvf B
boundary: [6, 5] critical? False in dgvf B
boundary: [5, 3] critical? False in dgvf B
boundary: [3, 0] critical? True in dgvf C
boundary: [3, 1] critical? False in dgvf A
boundary: [3, 6] critical? True in dgvf C
boundary: [6, 1] critical? False in dgvf A
boundary: [6, 4] critical? False in dgvf A
boundary: [4, 1] critical? True in dgvf C
boundary: [0, 1, 3] critical? False in dgvf B
boundary: [1, 3, 6] critical? False in dgvf B
boundary: [1, 4, 6] critical? Fals

# TO DO (updated sept 28)
- Data: reorder vertices in incr order ie 0 1 2 etc. then we don't have to deal with permutations and are also standard (otherwise boundary maps would change)
- Make importer function into function instead of in body of code
- There was some issue with the simplex trees not fully integrating with the classes (it didn't throw an error, it just was missing some finishing touches). Remember what this issue was and fix it.
- Get importer function to handle dgvf as well so we don't have to hardcode it.

# Usage Examples

In [9]:
print(V, "\n")
print(V['A'], "\n")

for simplex in simplices_list:
    print(simplex.entrancevals, simplex.boundary, simplex.init_index)
print("\n")

# this prints the number of i-dim simplices
input_complex.print_num_i_simplices(2)
print("\n")

input_complex.print_all_simplices()

{'A': [[1], [3, 1], [6, 1], [2], [4], [6, 4], [4, 7], [5], [6], [7]], 'B': [[0, 1], [0, 1, 3], [1, 3, 6], [1, 2], [2, 4], [1, 4, 6], [4, 6, 7], [5, 3], [6, 5], [7, 6]], 'C': [[0], [3, 0], [4, 1], [3], [3, 6]]} 

[[1], [3, 1], [6, 1], [2], [4], [6, 4], [4, 7], [5], [6], [7]] 

[0.0, 0.0] [0] 0
[1.0, 0.0] [1] 1
[2.0, 0.0] [2] 2
[0.0, 0.0] [3] 3
[2.0, 0.0] [4] 4
[0.0, 2.0] [5] 5
[0.0, 2.0] [6] 6
[3.0, 3.0] [7] 7
[0.0, 1.0] [0, 1] 8
[0.0, 2.0] [1, 2] 9
[0.0, 2.0] [2, 4] 10
[3.0, 3.0] [4, 7] 11
[3.0, 3.0] [7, 6] 12
[0.0, 2.0] [6, 5] 13
[0.0, 2.0] [5, 3] 14
[2.0, 0.0] [3, 0] 15
[2.0, 2.0] [3, 1] 16
[0.0, 2.0] [3, 6] 17
[2.0, 3.0] [6, 1] 18
[2.0, 3.0] [6, 4] 19
[2.0, 0.0] [4, 1] 20
[2.0, 2.0] [0, 1, 3] 21
[2.0, 3.0] [1, 3, 6] 22
[2.0, 3.0] [1, 4, 6] 23
[3.0, 3.0] [4, 6, 7] 24


given dim is  2  top dim is  2
number of 2 -simplices is  4


boundary: [0] critical? True in dgvf C
boundary: [1] critical? False in dgvf A
boundary: [2] critical? False in dgvf A
boundary: [3] critical? True in dgvf 